# HAR Model Training Notebook
This notebook builds, trains, and evaluates MLP and logistic regression models for the UCI HAR dataset, implements early stopping, and serializes weights for frontend inference.

In [1]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
import json

In [2]:
# load preprocessed data
train_df = pd.read_csv('train_processed.csv')
test_df = pd.read_csv('test_processed.csv')

In [3]:
# explore dfs
print('Train DataFrame shape:', train_df.shape)
print('Test DataFrame shape:', test_df.shape)
print('Train columns:', train_df.columns.tolist())
print('Test columns:', test_df.columns.tolist())
train_df.head(), test_df.head()

Train DataFrame shape: (113, 3)
Test DataFrame shape: (45, 3)
Train columns: ['mean', 'energy', 'entropy']
Test columns: ['mean', 'energy', 'entropy']


(       mean    energy    entropy
 0  0.109501  2.659607  24.802269
 1  0.233588  7.347243  42.604226
 2  0.273873  9.602077  45.398066
 3  0.273084  9.546216  45.369252
 4  0.269037  9.265970  45.209914,
        mean     energy    entropy
 0  0.118757   3.211997  25.370669
 1  0.255988   8.878185  43.561926
 2  0.279323  10.049428  45.487377
 3  0.265861   9.058485  45.061527
 4  0.274428   9.643114  45.415055)

In [8]:
#check for missing values, nans and duplicates

print("Missing values in train_df:\n", train_df.isnull().sum())
print("Number of duplicate rows in train_df:", train_df.duplicated().sum())
print("Duplicate rows in train_df:\n", train_df[train_df.duplicated()])

print("Missing values in test_df:\n", test_df.isnull().sum())
print("Number of duplicate rows in test_df:", test_df.duplicated().sum())
print("Duplicate rows in test_df:\n", test_df[test_df.duplicated()])

Missing values in train_df:
 mean       0
energy     0
entropy    0
dtype: int64
Number of duplicate rows in train_df: 0
Duplicate rows in train_df:
 Empty DataFrame
Columns: [mean, energy, entropy]
Index: []
Missing values in test_df:
 mean       0
energy     0
entropy    0
dtype: int64
Number of duplicate rows in test_df: 0
Duplicate rows in test_df:
 Empty DataFrame
Columns: [mean, energy, entropy]
Index: []


### So we have no missing, duplicate, or nan values.

In [5]:
# Model Training: MLP and Logistic Regression
# Check if 'label' column exists before proceeding
if 'label' in train_df.columns and 'label' in test_df.columns:
    X_train = train_df.drop('label', axis=1)
    y_train = train_df['label']
    X_test = test_df.drop('label', axis=1)
    y_test = test_df['label']

    # Logistic Regression Baseline
    logreg = LogisticRegression(max_iter=1000)
    logreg.fit(X_train, y_train)
    logreg_pred = logreg.predict(X_test)
    logreg_acc = accuracy_score(y_test, logreg_pred)
    print('Logistic Regression Accuracy:', logreg_acc)

    # MLP Classifier
    mlp = MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu', max_iter=100, early_stopping=True)
    mlp.fit(X_train, y_train)
    mlp_pred = mlp.predict(X_test)
    mlp_acc = accuracy_score(y_test, mlp_pred)
    print('MLP Accuracy:', mlp_acc)
else:
    print("'label' column not found in processed CSVs. Please check preprocessing and ensure labels are included.")

'label' column not found in processed CSVs. Please check preprocessing and ensure labels are included.


In [6]:
# Model Serialization
# Save MLP weights and biases
mlp_weights = {'coefs': [coef.tolist() for coef in mlp.coefs_], 'intercepts': [intercept.tolist() for intercept in mlp.intercepts_]}
with open('../../public/model/weights.json', 'w') as f:
    json.dump(mlp_weights, f)
print('Saved MLP weights to weights.json')

NameError: name 'mlp' is not defined

In [ ]:
# Fix for missing 'label' column
# If your processed CSVs do not include 'label', you must add it during preprocessing.
# For now, let's check columns and handle accordingly.

print('Train columns:', train_df.columns.tolist())
print('Test columns:', test_df.columns.tolist())

if 'label' not in train_df.columns:
    # Example: If labels are in a separate file or column, load and merge here
    print("'label' column missing. Please ensure your preprocessing includes activity labels.")
else:
    X_train = train_df.drop('label', axis=1)
    y_train = train_df['label']
    X_test = test_df.drop('label', axis=1)
    y_test = test_df['label']
    # ...existing code for model training...


In [ ]:
import Papa from 'papaparse';

async function loadTestSample(runInference: (features: number[]) => any) {
  const response = await fetch('/scripts/training/test_processed.csv');
  const csv = await response.text();
  const data = Papa.parse(csv, { header: true, skipEmptyLines: true }).data;
  const row = data[Math.floor(Math.random() * data.length)];
  const features = Object.values(row).slice(0, -1).map(Number); // adjust if label column is named differently
  const prediction = runInference(features);
  alert(`Predicted: ${prediction}, True: ${row['label'] || row['Activity']}`);
}